# D1 · Comparación inter-método (v4, por observable)

**Spec:** [`docs/spec_D1_v4_codex_method_comparison.md`](../docs/spec_D1_v4_codex_method_comparison.md)  |  **Bloque:** D · Método  |  **Run de este set:** `ROXs42Bb_realigned`

Compara los 6 métodos de extracción (C2–C6) por pares y banda sobre 33 controles, **separando continuo y líneas como observables distintos** (v4), y emite `recommended_method` con el árbol congelado (nunca fija el canónico).

| | |
|---|---|
| **Entrada** | C2/C3/C4 + C5/C6 (sgf/lpm) + G1 verdicts + predictor Ec. 1 de C5 |
| **Salida (QC/productos)** | `stages/stage_x10_qc.json` |
| **Consume aguas abajo** | D2 (consume el canónico de config) |


## Qué hace D1 v4 y cómo decide

D1 compara los métodos **por pares y por banda** con un **t control-centrado** (df = n_controles − 1): la diferencia del objeto contra la distribución de las diferencias de los 33 controles — referenciado a controles por construcción. Bandas B1–B6 (continuo) y LHα/LHβ/LOI (líneas); umbrales congelados p<0.0455 (divergente) y p<0.0027 (fuerte). Todo eso viene de v2/v3 y **no cambia**.

**Qué cambia en v4** ([`docs/spec_D1_v4_codex_method_comparison.md`](../docs/spec_D1_v4_codex_method_comparison.md), congelada 2026-07-29). La auditoría del cubo multiépoca encontró que v3 mezclaba tres observables distintos en un único estadístico:

| Defecto de v3 | Qué hace v4 |
|---|---|
| Aplicaba el throughput escalar **medido con inyecciones de Hα** a todas las bandas de continuo — una extrapolación cromática nunca medida, capaz de **crear** diferencias entre continuos que ya concordaban | B1–B6 se comparan sobre el flujo persistido tal cual, en la `SCALEREF` común: `comparison_mode = raw_total_continuum`, **throughput no aplicado** |
| Integraba las bandas de línea **sin restar continuo local**, así que la fila `LOI` podía estar dominada por el nivel de continuo y no por O I 8446 | Se resta la mediana móvil canónica de 80 Å antes de integrar, y el residual de línea se divide por el throughput **una sola vez**: `comparison_mode = local_continuum_subtracted_throughput_line` |
| Dejaba que **SGF y LPM** gobernaran `divergent_continuum` contra extractores de flujo total, cuando ambos eliminan o absorben el continuo **por construcción** | `CONTINUUM_METHODS = (aperture, optimal_ls, optimal_psfsub, psffit)`. SGF y LPM siguen en todas las tablas como diagnóstico (`role=diagnostic_noncomparable_continuum`) pero **no pueden activar** `divergent_continuum` |

El QC publica ahora `primary_pairs_continuum` y `primary_pairs_lines` por separado (`primary_pairs` se conserva como la unión, por compatibilidad), y el gate binomial de controles se calcula **por observable**: una fila diagnóstica ya no degrada un observable al que no pertenece.

**Por qué esto importa aquí y no es cosmético.** El veredicto v3 registrado para ROXs 12 B era `divergent_continuum`, y lo activaba B6 con t≈+14 de **psffit contra sgf/lpm** — exactamente los dos métodos que v4 excluye del veredicto de continuo. Con la política v4, esa señal pasa a ser diagnóstica y el continuo lo deciden los cuatro métodos comparables entre sí. **El veredicto puede cambiar al re-ejecutar**, y por eso la celda de abajo empieza declarando qué versión de QC está leyendo.

Lo que v4 **no** toca: `METHOD_ORDER`, las bandas, los umbrales, el scale-check, la jerarquía G1, el árbol de recomendación y la regla de que la elección canónica es humana.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x10_compare.sh --run-id $RUN
```

Ligero (~6 s).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_x10_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x10_compare.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_x10_qc.json', RUN_ID)
nb.show(qc, keys=['spec_version', 'verdict', 'action', 'recommended_method', 'primary_pairs_continuum', 'primary_pairs_lines', 'comparison_policy'], title='D1')


## Los chequeos del QC, en físico

D1 decide si dos métodos **discrepan de verdad** o solo por ruido. Sus chequeos vigilan las tres formas de engañarse en esa comparación: subestimar el ruido, comparar contra controles sucios y contar una corrección dos veces.

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v1_sigma_empirical_le_naive` | **¿Hay dispersión que los errores individuales no explican?** Exige `σ_empírico ≤ σ_ingenuo`, donde el ingenuo es `√(σ_i² + σ_j²)` (errores independientes). Dos métodos sobre el **mismo** cubo están positivamente correlacionados, así que la varianza de su diferencia debe ser *menor* que la ingenua: `σ_emp < σ_ing` es lo normal y no es un fallo. | Un σ empírico **mayor** que el ingenuo delata varianza extra —controles mal emparejados, una sistemática no modelada, o σ individuales subestimados— y el QC lo declara en `open_issues` con la celda concreta (par y banda) que la produce. |
| `v2_controls_clean` | **¿Los controles de los pares primarios son utilizables?** Ningún par primario degradado (por controles ausentes o inconsistentes). | El veredicto del par se marca degradado: no se puede afirmar que dos métodos difieran. |
| `v5_no_double_throughput` | **¿Se aplicó la corrección de throughput dos veces?** La corrección es interna a la comparación (solo en memoria); los productos en disco quedan SIN corregir y E3/G2 aplican la suya. | El flujo del compañero saldría corregido dos veces aguas abajo — un sesgo silencioso en Ṁ. |
| `v6_scale_check_ok` | **¿Están los dos métodos en la misma escala antes de restarlos?** | Se estaría midiendo una diferencia de convención, no de física. |
| `v7_t_calibration` | **¿El estadístico t está bien calibrado?** Fracción de pares de CONTROL que salen «divergentes» vs la fracción esperada: en posiciones sin fuente, los métodos solo pueden diferir por ruido, así que el ritmo de falsos positivos debe ser el nominal. | El umbral de «discrepan» no significa lo que dice. |


## Resultados que llevaron a la conclusión

Versión de QC, política de comparación, pares primarios **por observable** y t por banda de cada par primario. Si el QC en disco es todavía `D1_v3`, la celda lo dice y no reinterpreta esos números con la política nueva: un QC v3 se lee como v3.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('D1', 'stages/stage_x10_qc.json'):
        from scipy.stats import t as t_dist
        q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
        spec = q.get('spec_version')
        policy = q.get('comparison_policy')
        print(f"spec del QC en disco : {spec}")
        if spec != 'D1_v4':
            print('   ⚠ Este QC es ANTERIOR a la spec v4: sus B1-B6 llevan throughput de Hα\n'
                  '     aplicado y sus bandas de línea no tienen el continuo local restado.\n'
                  '     El veredicto de continuo pudo activarlo un par sgf/lpm, que v4 ya no\n'
                  '     deja votar. Hay que re-ejecutar D1 para leerlo con la política nueva.')
        else:
            print(f"   continuo : {policy['continuum_mode']}  sobre {policy['continuum_methods']}")
            print(f"   líneas   : {policy['line_mode']}  (ventana {policy['line_continuum_window_A']:.0f} Å)")
            for method, why in (q.get('excluded_from_continuum_verdict') or {}).items():
                print(f"   sin voto en continuo: {method} ({why})")
        print()
        print(f"veredicto : {q['verdict']}   ->  {q['action']}")
        print(f"motivo    : {q.get('reason')}")
        print(f"recomendado: {q['recommended_method']}   elegibles: {q['recommendation_rules']['eligible']}")
        print()
        # `primary_pairs_*` solo existen en v4; en v3 hay una única lista.
        groups = [(k, q[k]) for k in ('primary_pairs_continuum', 'primary_pairs_lines') if k in q]
        if not groups:
            groups = [('primary_pairs (v3, sin separar observable)', q['primary_pairs'])]
        st = q['statistics']; df = st['n_controls'] - 1
        T_DIV = t_dist.ppf(1 - st['p_divergent'] / 2, df)
        T_STR = t_dist.ppf(1 - st['p_strong'] / 2, df)
        for name, pairs in groups:
            print(f"{name}: {list(pairs)}")
        for name, pairs in groups:
            for pp in pairs:
                print(f'\nt control-centrado ({pp})  [{name.split()[0]}]:')
                for b, t in q['t_matrix'][pp].items():
                    if t is None: continue
                    flag = '  <-- FUERTE' if abs(t) > T_STR else ('  <- marginal' if abs(t) > T_DIV else '')
                    print(f'   {b:8s}: t = {t:+.2f}{flag}')


## Plot 1 — t control-centrado por par × banda (15 pares)

Del `t_matrix` del QC. Se marcan con `C` los pares que pueden votar el **continuo** (los cuatro de `CONTINUUM_METHODS`) y con `L` los que votan **líneas**; el resto están en la tabla como diagnóstico y no activan veredicto. Bajo v4 las columnas B1–B6 y las de línea ya no son la misma cantidad física: las primeras son flujo crudo integrado, las segundas residual de línea sobre continuo local.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import t as t_dist
    q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
    st = q['statistics']
    T_STR = t_dist.ppf(1 - st['p_strong'] / 2, st['n_controls'] - 1)
    tm = q['t_matrix']
    cont = list(q.get('primary_pairs_continuum', []))
    line = list(q.get('primary_pairs_lines', []))
    primary = list(dict.fromkeys(cont + line)) or list(q['primary_pairs'])
    pairs = primary + [p for p in tm if p not in primary]
    bands = list(tm[pairs[0]].keys())
    M = np.array([[np.nan if tm[p].get(b) is None else tm[p][b] for b in bands] for p in pairs])
    fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(pairs) + 1.8))
    im = ax.imshow(M, cmap='RdBu_r', vmin=-6, vmax=6, aspect='auto')
    ax.set_xticks(range(len(bands))); ax.set_xticklabels(bands)
    def _role(p):
        tag = ('C' if p in cont else '') + ('L' if p in line else '')
        return (tag or ('*' if p in primary else '  ')).ljust(2) + ' '
    labels = [_role(p) + p.replace('_vs_', ' vs ') for p in pairs]
    ax.set_yticks(range(len(pairs))); ax.set_yticklabels(labels, fontsize=7)
    for i in range(len(pairs)):
        for j in range(len(bands)):
            v = M[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=6, color='k' if abs(v) < 4 else 'w')
    ax.set_title(f"D1 {q.get('spec_version','?')} · t por par × banda "
                 f"(C=vota continuo, L=vota líneas; |t|>{T_STR:.2f} fuerte)")
    fig.colorbar(im, label='t'); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd1_compare'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'tmatrix.png', dpi=110); print('figura ->', outdir / 'tmatrix.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — los dos observables por separado

Arriba, las bandas de **continuo** B1–B6 con solo los pares que pueden votarlas; abajo, las de **línea** con sus pares primarios. Es la figura que hace visible el cambio de v4: si una divergencia vive únicamente en el panel de continuo con pares que ya no votan, deja de ser un veredicto y pasa a ser un diagnóstico. Las líneas punteadas son los umbrales divergente y fuerte.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import t as t_dist
    q = nb.load_qc('stages/stage_x10_qc.json', RUN_ID)
    st = q['statistics']; df = st['n_controls'] - 1
    t_div = t_dist.ppf(1 - st['p_divergent'] / 2, df)
    t_str = t_dist.ppf(1 - st['p_strong'] / 2, df)
    kinds = {b['name']: b['kind'] for b in q['bands']}
    all_bands = list(q['t_matrix'][list(q['t_matrix'])[0]].keys())
    cbands = [b for b in all_bands if kinds.get(b) == 'continuum']
    lbands = [b for b in all_bands if kinds.get(b) == 'line']
    if 'primary_pairs_continuum' in q:
        panels = [('continuo (B1-B6) · pares con voto', cbands, list(q['primary_pairs_continuum'])),
                  ('líneas · pares con voto', lbands, list(q['primary_pairs_lines']))]
    else:
        # QC v3: no existe la separación por observable. No se inventa: un único
        # panel con todos los pares primarios, y el título lo dice.
        panels = [('QC v3 · sin separar por observable (todos los pares primarios)',
                   all_bands, list(q['primary_pairs']))]
    fig, axes = plt.subplots(len(panels), 1, figsize=(10, 4.0 * len(panels)), squeeze=False)
    for ax, (title, bands, pairs) in zip(axes[:, 0], panels):
        if not bands or not pairs:
            ax.text(0.5, 0.5, 'sin pares con voto para este observable', ha='center',
                    va='center', transform=ax.transAxes, color='#8a8a8a')
            ax.set_title(f'{title} — sin veredicto'); ax.set_xticks([]); continue
        # Un punto por par y banda: aguanta 15 pares, que un agrupado de barras no.
        rng = np.random.default_rng(0)
        for j, b in enumerate(bands):
            vals = [(pp, q['t_matrix'].get(pp, {}).get(b)) for pp in pairs]
            vals = [(pp, t) for pp, t in vals if t is not None]
            # anota solo los 2 |t| mayores de cada banda: con 15 pares, anotarlos
            # todos tapa la figura y no se lee ninguno
            top = {pp for pp, _ in sorted(vals, key=lambda kv: -abs(kv[1]))[:2]}
            for pp, t in vals:
                strong = abs(t) > t_str
                ax.plot(j + rng.uniform(-0.18, 0.18), t, 'o', ms=5,
                        color='tab:red' if strong else ('tab:orange' if abs(t) > t_div else 'tab:blue'),
                        alpha=0.85)
                if strong and pp in top:
                    ax.annotate(pp.replace('_vs_', '/'), (j, t), fontsize=5.5,
                                xytext=(7, 0), textcoords='offset points', va='center')
        for s in (t_div, t_str):
            ax.axhline(s, color='k', ls=':', lw=0.8); ax.axhline(-s, color='k', ls=':', lw=0.8)
        ax.axhline(0, color='k', lw=0.6)
        # symlog: deja ver a la vez el umbral (~2-3) y las t de tres cifras
        ax.set_yscale('symlog', linthresh=max(t_str, 1.0))
        ax.set_xticks(range(len(bands))); ax.set_xticklabels(bands)
        ax.set_ylabel('t control-centrado (symlog)')
        ax.set_title(f'{title} · {len(pairs)} par(es)', fontsize=10)
    fig.suptitle(f"{q.get('spec_version','D1 ?')} · veredicto {q['verdict']} "
                 f"(rojo = fuerte, naranja = divergente)", y=0.995)
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd1_compare'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'primary_pairs.png', dpi=110); print('figura ->', outdir / 'primary_pairs.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Separación por observable (v4)**: el continuo lo deciden solo `aperture`, `optimal_ls`, `optimal_psfsub` y `psffit`; SGF y LPM quedan como diagnóstico porque anulan o absorben el continuo por construcción. Las líneas las deciden los seis, tras restar continuo local. · [`docs/spec_D1_v4_codex_method_comparison.md`](../docs/spec_D1_v4_codex_method_comparison.md)
- **El throughput de Hα ya no toca B1–B6.** Aplicarlo a bandas de continuo era una extrapolación cromática no medida que podía fabricar divergencias; ahora se aplica exactamente una vez y solo al residual de línea. · [`docs/spec_D1_v4_codex_method_comparison.md`](../docs/spec_D1_v4_codex_method_comparison.md)
- **METHOD_ORDER de 6** con la familia espectral (Julo+25); pares primarios = validados por G1; bandas, umbrales, scale-check y protocolo anti cherry-picking **sin cambios** desde v2/v3. · [`docs/spec_D1_v3_codex_method_comparison.md`](../docs/spec_D1_v3_codex_method_comparison.md)
- **optimal_psfsub → rejected** (histórico v3): su T venía de un E4 pre-consolidación Psfao; verificado con worktree HEAD que el cambio no provenía del código nuevo. Revisar contra el E4 v2 actual al re-ejecutar.
- **El usuario mantuvo psffit como canónico** (2026-07-15). Esa decisión sigue en pie y D1 no la toca: el árbol solo recomienda con veredicto `consistent` y la elección canónica es humana. · [`docs/2026-07-09_d1_canonical_method_decision.md`](../docs/2026-07-09_d1_canonical_method_decision.md)


## Conclusión (pendiente de re-ejecución)

**La spec v4 está congelada y el código la implementa (`SPEC_VERSION = D1_v4`), pero el veredicto de este objeto todavía no se ha vuelto a calcular con ella.** La celda de evidencia declara arriba qué versión de QC estás leyendo; no des por bueno un veredicto v3 como si fuera v4.

- **Lo que había (v3, 2026-07-15):** `divergent_continuum` con 3 pares primarios limpios (psffit–sgf, psffit–lpm, sgf–lpm), activado por B6 con t≈+14 de psffit contra la familia espectral. **Bajo v4 esos tres pares ya no votan el continuo**: dos de sus miembros son sgf y lpm.
- **Qué esperar al re-ejecutar:** el continuo pasa a decidirse entre `aperture/optimal_ls/optimal_psfsub/psffit` sobre flujo crudo, y las líneas sobre residual con continuo local restado. El veredicto puede cambiar; la firma B6 no desaparece, se reclasifica como diagnóstico.
- **Estadístico (sin cambios):** t control-centrado, 33 controles (df=32), corr_length 2.94 canales (de G1).
- **Downstream:** D2 calibró los 6 métodos; E1 re-confirmó la no-detección con 6 métodos. Si el veredicto de D1 cambia, hay que revisar qué arrastra hacia D2.
- Todo provisional hasta cerrar el A-block.
